# 🏭 [2026 Edge AI Challenge] 베이스라인 코드
본 노트북은 스마트 팩토리 철강 표면 결함 탐지 대회를 위한 기본 뼈대 코드입니다.
주어진 데이터를 활용해 모델을 훈련하고, 테스트 데이터에 대한 추론 시간을 측정하여 최종 제출용 `submission.csv` 파일을 생성합니다.

In [ ]:
import os
import time
import random
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset

# 시드 고정 (재현성 확보)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

# ==========================================
# 🌐 데이터셋 경로 설정 (🚨 반드시 본인 환경에 맞게 수정하세요!)
# ==========================================
# [1] Colab 또는 Local에서 실행할 경우: 데이터가 저장된 폴더 경로를 입력하세요.
COLAB_LOCAL_DATA_PATH = './NEU-DET'
BASE_DIR = COLAB_LOCAL_DATA_PATH
SAVE_DIR = './'


# 하위 폴더 경로 설정 (압축 푼 폴더 구조에 맞춰 필요시 수정)
TRAIN_DIR = os.path.join(BASE_DIR, 'train', 'images')
VAL_DIR = os.path.join(BASE_DIR, 'validation', 'images')
TEST_DIR = os.path.join(BASE_DIR, 'test', 'images')

print(f"📂 베이스 경로: {BASE_DIR}")
print(f"📂 저장 경로: {SAVE_DIR}")

📂 베이스 경로: ./NEU-DET
📂 저장 경로: ./


# Data Loader

In [2]:
# ==========================================
# 1. 커스텀 데이터 증강 (모션 블러)
# ==========================================
class RandomConveyorBeltMotionBlur(object):
    def __init__(self, kernel_size=21, p=0.5):
        self.kernel_size = kernel_size
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img
        img_np = np.array(img)
        kernel = np.zeros((self.kernel_size, self.kernel_size))
        kernel[int((self.kernel_size - 1)/2), :] = np.ones(self.kernel_size)
        kernel /= self.kernel_size
        blurred = cv2.filter2D(img_np, -1, kernel)
        return Image.fromarray(blurred)

# ==========================================
# 2. 트랜스폼 및 Dataset 세팅
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    RandomConveyorBeltMotionBlur(kernel_size=21, p=0.5), # 훈련 시 노이즈 예방주사
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 캐글 제출용 Test Dataset 클래스 (파일명 추출용)
class TestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.img_names = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, img_name

# Data Loaders
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

NUM_CLASSES = len(train_dataset.classes)
print(f"✅ 클래스 개수: {NUM_CLASSES}")

✅ 클래스 개수: 6


# Model Definition

In [ ]:
# ==========================================
# 💡 모델 세팅
# ==========================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 현재 디바이스: {DEVICE}")

# 베이스라인은 ResNet-50을 사용합니다.
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

# Training Loop

In [4]:
# Loss
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ==========================================
# 🚀 학습 루프 (Epoch 5 예시)
# ==========================================
EPOCHS = 5
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # 검증
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    print(f" -> Val Accuracy: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

print("🎉 학습 완료!")

✅ 현재 디바이스: cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 197MB/s]
Epoch 1/5 [Train]: 100%|██████████| 45/45 [00:26<00:00,  1.69it/s]


 -> Val Accuracy: 71.11%


Epoch 2/5 [Train]: 100%|██████████| 45/45 [00:25<00:00,  1.80it/s]


 -> Val Accuracy: 70.56%


...

 -> Val Accuracy: 65.00%


Epoch 4/5 [Train]: 100%|██████████| 45/45 [00:26<00:00,  1.69it/s]


 -> Val Accuracy: 59.44%


Epoch 5/5 [Train]: 100%|██████████| 45/45 [00:27<00:00,  1.62it/s]


 -> Val Accuracy: 85.56%
🎉 학습 완료!


# Submission
## 유의사항 , submission 파일 생성은 꼭 cpu를 이용해서 진행하시면 됩니다.
### Colab - [런타임 유형 변경] - [cpu]

In [ ]:
# ==========================================
# ⏱️ 추론 시간 측정 및 submission.csv 생성
# ==========================================
# 모델 로드
DEVICE = 'cpu'
model.load_state_dict(torch.load('/content/Best Model.pth',map_location=DEVICE))
model.eval()
################################################



# Test 데이터 로더 준비
test_dataset = TestDataset(TEST_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

predictions = []
image_ids = []

print("🚀 추론을 시작합니다...")

# --- ⏱️ 시간 측정 시작 ---
start_time = time.time()

with torch.no_grad():
    for images, img_names in tqdm(test_loader, desc="Inference"):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        predictions.extend(preds.cpu().numpy())
        image_ids.extend(img_names)

# --- ⏱️ 시간 측정 종료 ---
end_time = time.time()
total_inference_time = end_time - start_time

print(f"✅ 추론 완료! 총 소요 시간: {total_inference_time:.2f}초")

# ==========================================
# 📝 제출용 CSV 저장
# ==========================================
submission_df = pd.DataFrame({
    'Id': image_ids,
    'Expected': predictions,
    'inference_time_sec': round(total_inference_time, 2)
})

submission_path = os.path.join(SAVE_DIR, 'submission.csv')
submission_df.to_csv(submission_path, index=False)

print(f"🎉 제출 파일 저장 완료: {submission_path}")
submission_df.head()